# Hephaestus — CVE Analysis Automaton (LoRA, Qwen2.5-3B)
Model: Qwen/Qwen2.5-3B-Instruct  |  Method: QLoRA + SFT (plain transformers)  |  GPU: P100 (sm_60, cu121)

Closes Hephaestus v0.2 Plan 1.5. Uses the **verified Kaggle P100 recipe**:
torch 2.3.1+cu121, trl 0.8.6 (--no-deps), transformers 4.46.3, peft 0.13.2, bitsandbytes 0.46.1.
NO unsloth (its unsloth-zoo dep is missing on PyPI; and the cu117 torch wheel is not
installable on current Kaggle runtimes). Kaggle's default cu128 crashes on P100 (sm_70+),
so we pin cu121.

Dataset: `harinpurumandla/cve-severity-dataset` (PUBLIC). ~200k CVE entries, columns
`cve_id`, `description`, `severity` (CRITICAL/HIGH/MEDIUM/LOW), train/val/test splits.

Final push to `Yusif-v/hephaestus-cve-analyzer` needs a Kaggle secret `HF_TOKEN`;
if unset, training + eval still run and the upload is skipped.

Kaggle account: **yusifovtelman** (not GitHub yusif-v).

In [ ]:
# --- 0. Secrets / env ---
import os
from kaggle_secrets import UserSecretsClient

HF_TOKEN = None
try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN loaded from Kaggle secret')
except Exception as e:
    print(f'No Kaggle secret HF_TOKEN ({e}); training+eval will run, but the HF upload will be skipped.')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
# --- 1. Install torch 2.3.1 + cu121 (P100 sm_60) + plain transformers stack ---
# VERIFIED P100 recipe. AVOID unsloth (unsloth-zoo missing on PyPI); avoid cu117 (wheel
# not installable on current Kaggle runtime); avoid cu128 (crash on P100 sm_60).
%pip install -q torch==2.3.1+cu121 --index-url https://download.pytorch.org/whl/cu121
%pip install -q --no-deps trl==0.8.6
%pip install -q transformers==4.46.3 peft==0.13.2 accelerate datasets bitsandbytes==0.46.1

In [ ]:
# --- 2. Load + inspect the (PUBLIC) CVE dataset (with schema guard) ---
import pandas as pd
from datasets import load_dataset

DS = 'harinpurumandla/cve-severity-dataset'
ds = load_dataset(DS)  # public — no token needed
print('Splits:', list(ds.keys()))
for split in ds:
    print(f'  {split}: {len(ds[split])} rows')

sample = ds['train'][0]
print('Columns:', list(sample.keys()))
print('Sample:', {k: str(v)[:120] for k, v in sample.items()})
print('Severity distribution (train):', ds['train'].to_pandas()['severity'].value_counts().to_dict())

# GUARD: we expect cve_id / description / severity.
expected = {'cve_id', 'description', 'severity'}
have = set(sample.keys())
missing = expected - have
assert not missing, f'SCHEMA MISMATCH: missing {missing}. Got {have}. Fix the converter cell.'
print('Schema OK:', sorted(have))

In [ ]:
# --- 3. Convert to messages (mirrors hephaestus/loader.py CVE branch) ---
import json, re

CLASSES = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
SYSTEM = ('You are a cybersecurity expert specializing in CVE analysis. '
          'Given a CVE description, classify its severity as CRITICAL, HIGH, MEDIUM, or LOW, '
          'and provide a brief risk assessment.')

def to_messages(item):
    cve_id = item.get('cve_id', 'Unknown')
    description = item.get('description', item.get('text', ''))
    severity = str(item.get('severity', item.get('label', ''))).strip().upper()
    if severity not in CLASSES:
        sev_map = {'CRIT': 'CRITICAL', 'C': 'CRITICAL', 'H': 'HIGH', 'M': 'MEDIUM', 'L': 'LOW'}
        severity = sev_map.get(severity[:1], 'MEDIUM')
    user = f'Analyze this CVE:\n\nCVE ID: {cve_id}\nDescription: {description}'
    return [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': f'{severity}. ' + ('Requires immediate attention.' if severity in ('CRITICAL','HIGH') else 'Monitor and patch as routine.')},
    ]

MAX_TRAIN = 50000
raw_train = ds['train'].select(range(min(MAX_TRAIN, len(ds['train']))))
raw_test = ds['test']  # held-out 2025 CVEs
train_msgs = [{'messages': to_messages(r)} for r in raw_train]
test_msgs = [{'messages': to_messages(r)} for r in raw_test.select(range(min(1000, len(raw_test))))]

assert train_msgs and test_msgs, 'Produced empty train/test — converter or split wrong.'
print(f'Converted: train={len(train_msgs)} test={len(test_msgs)}')
print('Example:', json.dumps(train_msgs[0], indent=2)[:400])

In [ ]:
# --- 4. Build HF dataset from messages ---
from datasets import Dataset
train_ds = Dataset.from_list(train_msgs)
test_ds = Dataset.from_list(test_msgs)
print('train', len(train_ds), 'test', len(test_ds))

In [ ]:
# --- 5. Load Qwen2.5-3B with plain transformers + 4-bit (NO unsloth) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ = 2048
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ  # cap; avoids SFTTrainer max_seq_length kwarg drift
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16)
lora = LoraConfig(r=256, lora_alpha=512, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora)
print('LoRA applied')

In [ ]:
# --- 6. Train ---
# Use the version-stable SFTTrainer signature: tokenizer= + dataset_text_field= + args=.
# Do NOT pass max_seq_length= (kwarg name drifts across TRL versions); cap via tokenizer above.
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=test_ds,
    dataset_text_field='messages',
    args=TrainingArguments(
        per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=2, warmup_steps=20, max_steps=200,
        learning_rate=2e-5, fp16=True, logging_steps=10,
        eval_strategy='steps', eval_steps=40, save_strategy='steps', save_steps=40,
        load_best_model_at_end=True, metric_for_best_model='eval_loss',
        output_dir='/kaggle/working/outputs', report_to='none',
    ),
)
trainer.train()

In [ ]:
# --- 7. Evaluate (multiclass, mirrors hephaestus/evaluator.py) ---
import re
from collections import defaultdict
model.eval()

def extract_label(text, classes):
    t = text.upper().strip()
    for c in sorted(classes, key=len, reverse=True):
        if re.search(r'\b' + re.escape(c) + r'\b', t):
            return c
    return classes[0]

per_tp, per_fp, per_fn, per_corr, per_tot = (defaultdict(int) for _ in range(5))
dev = next(model.parameters()).device
for item in test_msgs:
    msgs = item['messages']
    expected = next(m['content'] for m in reversed(msgs) if m['role']=='assistant')
    exp = extract_label(expected, CLASSES); per_tot[exp]+=1
    prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(dev)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=64, do_sample=False)
    resp = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_label(resp, CLASSES)
    if pred==exp: per_corr[exp]+=1; per_tp[pred]+=1
    else: per_fn[exp]+=1; per_fp[pred]+=1

n=len(test_msgs); correct=sum(per_corr.values())
acc=correct/n
ps=[per_tp[c]/(per_tp[c]+per_fp[c]) if per_tp[c]+per_fp[c] else 0 for c in CLASSES]
rs=[per_tp[c]/(per_tp[c]+per_fn[c]) if per_tp[c]+per_fn[c] else 0 for c in CLASSES]
fs=[2*p*r/(p+r) if p+r else 0 for p,r in zip(ps,rs)]
print(f'Accuracy: {acc*100:.1f}% | Macro-F1: {sum(fs)/4*100:.1f}%')
for c in CLASSES:
    a = per_corr[c]/per_tot[c] if per_tot[c] else 0
    print(f'  {c}: acc={a*100:.1f}% (n={per_tot[c]})')
GATE=0.95
print(f'\nQuality Gate (95%): {"PASS" if acc>=GATE else "FAIL"}')

In [ ]:
# --- 8. Merge + push to HuggingFace as Yusif-v/hephaestus-cve-analyzer ---
if not HF_TOKEN:
    print('SKIPPING HF upload: no HF_TOKEN secret set. Set a Kaggle secret HF_TOKEN and re-run cell 8 to upload.')
else:
    merged = model.merge_and_unload()
    merged.push_to_hub('Yusif-v/hephaestus-cve-analyzer', token=HF_TOKEN)
    tokenizer.push_to_hub('Yusif-v/hephaestus-cve-analyzer', token=HF_TOKEN)
    print('Pushed Yusif-v/hephaestus-cve-analyzer')